In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.utils import class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, roc_curve
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, Dense, Dropout, BatchNormalization,
    Bidirectional, LSTM, GlobalAveragePooling1D, MultiHeadAttention, LayerNormalization, Add,
    Concatenate
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt

def one_hot_encode(seq):
    mapping = {'A': [1, 0, 0, 0],
               'C': [0, 1, 0, 0],
               'G': [0, 0, 1, 0],
               'T': [0, 0, 0, 1]}
    return np.array([mapping.get(base, [0, 0, 0, 0]) for base in seq])

def preprocess_all_dfs(excel_path, sheet_names):
    dfs = [pd.read_excel(excel_path, sheet_name=sheet) for sheet in sheet_names]
    all_chromosomes = pd.concat([df["Chromosome"] for df in dfs], ignore_index=True).to_frame()
    chrom_encoder = OneHotEncoder(sparse_output=False)
    chrom_encoder.fit(all_chromosomes)
    processed_data = {}
    for sheet, df in zip(sheet_names, dfs):
        df_filtered = df[(df['Normalized efficacy'] > 0.35) | (df['Normalized efficacy'] < 0.15)].copy()
        df_filtered['label'] = (df_filtered['Normalized efficacy'] > 0.35).astype(int)
        X_seq = np.array([one_hot_encode(seq) for seq in df_filtered["sgRNA"]])
        y = df_filtered["label"].values
        chrom_encoded = chrom_encoder.transform(df_filtered[["Chromosome"]])
        start_norm = (df_filtered["Start"] - df_filtered["Start"].min()) / (df_filtered["Start"].max() - df_filtered["Start"].min())
        end_norm = (df_filtered["End"] - df_filtered["End"].min()) / (df_filtered["End"].max() - df_filtered["End"].min())
        X_meta = np.concatenate([chrom_encoded, start_norm.values.reshape(-1, 1), end_norm.values.reshape(-1, 1)], axis=1)
        processed_data[sheet] = (X_seq, X_meta, y)
    return processed_data, chrom_encoder

def build_model(input_seq_shape, input_meta_shape):
    input_seq = Input(shape=input_seq_shape, name='sequence_input')
    input_meta = Input(shape=input_meta_shape, name='meta_input')
    x = Conv1D(filters=256, kernel_size=7, activation="relu", padding='same')(input_seq)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Conv1D(filters=256, kernel_size=5, activation="relu", padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Dropout(0.4)(x)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = Dropout(0.4)(x)
    attn_out = MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
    x = Add()([x, attn_out])
    x = LayerNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    combined = Concatenate()([x, input_meta])
    x = Dense(128, activation="relu")(combined)
    x = Dropout(0.5)(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation="sigmoid")(x)
    model = Model(inputs=[input_seq, input_meta], outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    return model

def main():
    excel_path = "/13059_2018_1459_MOESM5_ESM (1).xlsx"
    sheet_names = ["hela", "hct116", "hl60", "hek293t"]
    processed_data, chrom_encoder = preprocess_all_dfs(excel_path, sheet_names)
    roc_data = {}

    for test_cell in sheet_names:
        print(f"\n=== Training on all except {test_cell}, Testing on {test_cell} ===")
        train_cells = [c for c in sheet_names if c != test_cell]
        X_seq_full = np.concatenate([processed_data[c][0] for c in train_cells])
        X_meta_full = np.concatenate([processed_data[c][1] for c in train_cells])
        y_full = np.concatenate([processed_data[c][2] for c in train_cells])
        X_seq_train, X_seq_val, X_meta_train, X_meta_val, y_train, y_val = train_test_split(
            X_seq_full, X_meta_full, y_full, test_size=0.2, stratify=y_full, random_state=42
        )
        X_seq_test, X_meta_test, y_test = processed_data[test_cell]
        class_weights_dict = dict(enumerate(
            class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        ))

        model = build_model(input_seq_shape=X_seq_train.shape[1:], input_meta_shape=X_meta_train.shape[1:])
        early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor="val_loss", patience=5, factor=0.5, min_lr=1e-6)
        model.fit(
            [X_seq_train, X_meta_train], y_train,
            validation_data=([X_seq_val, X_meta_val], y_val),
            epochs=50,
            batch_size=32,
            callbacks=[early_stop, reduce_lr],
            class_weight=class_weights_dict,
            verbose = 0
        )

        y_probs = model.predict([X_seq_test, X_meta_test]).flatten()
        y_pred = (y_probs >= 0.5).astype(int)
        print("\n[Test Set Performance (held-out cell line)]")
        print(classification_report(y_test, y_pred))
        print(f"ROC AUC Score (test): {roc_auc_score(y_test, y_probs):.4f}")
        fpr, tpr, _ = roc_curve(y_test, y_probs)
        auc_score = roc_auc_score(y_test, y_probs)
        roc_data[test_cell.upper()] = {"fpr": fpr, "tpr": tpr, "auc": auc_score}

    plt.figure(figsize=(8, 6))
    for cell, stats in roc_data.items():
        plt.plot(stats["fpr"], stats["tpr"], label=f"{cell} (AUC = {stats['auc']:.2f})")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves (Leave-One-Cell-Line-Out)')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    cell_names = list(roc_data.keys())
    auc_scores = [roc_data[c]["auc"] for c in cell_names]
    plt.figure(figsize=(6, 4))
    bars = plt.bar(cell_names, auc_scores, color='skyblue')
    plt.ylim(0, 1.0)
    plt.ylabel('AUC Score')
    plt.title('AUC by Cell Line')
    for bar, score in zip(bars, auc_scores):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{score:.2f}', ha='center')
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()





FileNotFoundError: [Errno 2] No such file or directory: '/13059_2018_1459_MOESM5_ESM (1).xlsx'